# Clustering Pipeline

## What problem are we solving?

**Clustering** is a form of *unsupervised learning*: we are given a pile of data points with **no labels** telling us what each one "is", and we want an algorithm to discover groups (clusters) of similar points on its own. Contrast this with *supervised learning* (e.g. classification), where every training example already comes with the correct answer.

This notebook builds three different families of clustering algorithm from the ground up, then compares them against their production-grade `scikit-learn` / `scipy` equivalents:

| Family | Idea in one sentence | Tasks |
| --- | --- | --- |
| **K-means** (hard clustering) | Every point belongs to exactly one cluster, defined by the nearest centroid. | 0–3, 10 |
| **Gaussian Mixture Model / EM** (soft clustering) | Every point has a *probability* of belonging to each cluster, modeled as a mix of overlapping bell-curves. | 4–9, 11 |
| **Agglomerative / hierarchical clustering** | Start with every point as its own cluster and repeatedly merge the closest pair, building a tree. | 12 |

## How to read each section

Every section below follows the same structure, in a "learn by example" style:

- **What it does** — a plain-English definition.
- **How it works** — the algorithm as a short numbered recipe.
- **Key lines explained** — the actual code from this project, annotated line by line.
- **Where this fits in the pipeline** — how it depends on the *previous* step and sets up the *next* one.
- **Professor's note** — a beginner-friendly aside: common pitfalls, terminology you'll see elsewhere, or a link between the toy code here and real-world tooling.
- **References** — official documentation / reputable sources to read further.

Run this notebook from the `unsupervised_learning/clustering` directory so the `__import__('N-module')` calls can find the numbered solution files.

> **Beginner tip:** you will see `__import__('0-initialize').initialize` instead of a normal `from module import function`. That's because these module filenames start with a digit (`0-initialize.py`), which is **not a legal Python identifier** — `import 0-initialize` would be a syntax error. `__import__()` is the built-in, string-based, low-level function that Python's own `import` statement calls internally, so passing it a string sidesteps the naming restriction. See the [Python official docs on `__import__()`](https://docs.python.org/3/library/functions.html#import__).

## Learning Objectives

### 1. What is a multimodal distribution?
A **multimodal distribution** is a probability distribution with **two or more distinct peaks** (local modes) in its probability density function.
- Unlike a *unimodal* distribution (like a standard single Gaussian curve with one peak), a multimodal distribution indicates that the data comes from multiple distinct subpopulations or regimes.
- *Example:* Human heights in a mixed population typically form a bimodal distribution (one peak for females, one for males). In unsupervised learning, multimodal data is a natural candidate to be modeled using mixture models.

---

### 2. What is a cluster?
A **cluster** is a grouping or collection of data points that share similar characteristics, features, or spatial proximity.
- The fundamental principle of a cluster is:
  - **High intra-cluster similarity:** Points within the same cluster are close and similar to one another.
  - **Low inter-cluster similarity:** Points in different clusters are well separated.

---

### 3. What is cluster analysis?
**Cluster analysis** (clustering) is the unsupervised machine learning task of grouping a set of unlabeled objects so that objects in the same group are more similar to each other than to those in other groups.
- Because there are **no target labels** provided during training, cluster analysis discovers hidden patterns, groupings, and taxonomic structures directly from the data.
- Widely used for customer segmentation, anomaly detection, image compression, document organization, and bioinformatics.

---

### 4. What is "soft" vs "hard" clustering?
- **Hard Clustering:** Each data point is assigned strictly and exclusively to **exactly one** cluster:
  - Binary membership: $w_{ij} \in \{0, 1\}$.
  - A point either belongs 100% to cluster $j$ or 0%. There is no ambiguity or partial assignment.
  - *Example:* Standard **K-means**.
- **Soft (Fuzzy / Probabilistic) Clustering:** Each data point has a **probability** or continuous degree of responsibility belonging to each cluster:
  - Continuous membership: $w_{ij} \in [0, 1]$, with $\sum_{j=1}^k w_{ij} = 1$.
  - Points near cluster boundaries have fractional memberships across multiple clusters (e.g. 60% Cluster A, 40% Cluster B).
  - *Example:* **Gaussian Mixture Models (GMM)** trained via Expectation-Maximization.

---

### 5. What is K-means clustering?
**K-means** is an iterative, centroid-based partitioning algorithm that divides $n$ data points into $k$ non-overlapping clusters.
- **Objective:** Minimize the total within-cluster variance (Within-Cluster Sum of Squares, or inertia):
  $$J = \sum_{j=1}^k \sum_{x_i \in C_j} \|x_i - \mu_j\|^2$$
- **Algorithm (Lloyd's Algorithm):**
  1. Initialize $k$ centroids $\mu_1, \dots, \mu_k$ (e.g. uniformly within data bounds or using k-means++).
  2. **Assignment Step:** Assign each data point to its nearest centroid using Euclidean distance.
  3. **Update Step:** Recompute each centroid as the arithmetic mean of all points assigned to that cluster:
     $$\mu_j = \frac{1}{|C_j|} \sum_{x_i \in C_j} x_i$$
  4. Repeat assignment and update steps until centroids stop moving (convergence) or max iterations is reached.

---

### 6. What are mixture models?
A **mixture model** is a probabilistic model that represents the overall probability density of data as a weighted sum of multiple component distributions:
$$p(x) = \sum_{j=1}^k \pi_j p_j(x \mid \theta_j)$$
- $k$: Number of mixture components.
- $\pi_j$: Mixing weights (priors), where $0 \le \pi_j \le 1$ and $\sum_{j=1}^k \pi_j = 1$.
- $p_j(x \mid \theta_j)$: Probability density function of component $j$.
- Mixture models can approximate virtually any complex, multimodal continuous density function.

---

### 7. What is a Gaussian Mixture Model (GMM)?
A **Gaussian Mixture Model (GMM)** is a mixture model where each individual component is a multivariate normal (Gaussian) distribution:
$$p(x) = \sum_{j=1}^k \pi_j \mathcal{N}(x \mid \mu_j, \Sigma_j)$$
- $\pi_j$: Prior probability of cluster $j$.
- $\mu_j$: Centroid mean vector of cluster $j$.
- $\Sigma_j$: Covariance matrix of cluster $j$.
- Unlike K-means (which implicitly assumes spherical clusters of equal size), GMM covariance matrices allow each cluster to take on arbitrary **ellipsoidal shapes, sizes, and orientations**.

---

### 8. What is the Expectation-Maximization (EM) algorithm?
The **Expectation-Maximization (EM)** algorithm is an iterative optimization framework to find Maximum Likelihood Estimates (MLE) of model parameters in problems with **latent (hidden / unobserved) variables**.
- In clustering, the observed data is $X$, and the latent variable is the true cluster assignment of each point.
- Because directly maximizing the marginal log-likelihood $\ln p(X)$ is intractable, EM alternates between:
  1. **E-step (Expectation):** Computes the expected values of the latent variables given current parameter estimates (calculates responsibilities $\gamma_{ij}$).
  2. **M-step (Maximization):** Updates model parameters ($\pi, \mu, \Sigma$) to maximize the expected complete-data log-likelihood found in the E-step.
- Every EM cycle is mathematically guaranteed to increase (or maintain) the observed data log-likelihood until reaching a local optimum.

---

### 9. How to implement the EM algorithm for GMMs?
1. **Initialization:**
   - Initialize priors: $\pi_j = \frac{1}{k}$.
   - Initialize means: $\mu_j$ using K-means centroids.
   - Initialize covariances: $\Sigma_j = I_d$ (identity matrices).
2. **Expectation Step (E-step):**
   - For each point $x_i$ and cluster $j$, compute the Gaussian PDF $p(x_i \mid \mu_j, \Sigma_j)$.
   - Compute posterior responsibilities:
     $$\gamma_{ij} = \frac{\pi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}{\sum_{l=1}^k \pi_l \mathcal{N}(x_i \mid \mu_l, \Sigma_l)}$$
   - Compute total log-likelihood:
     $$\ln L = \sum_{i=1}^n \ln \left(\sum_{j=1}^k \pi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)\right)$$
3. **Maximization Step (M-step):**
   - $N_j = \sum_{i=1}^n \gamma_{ij}$ (effective number of points in cluster $j$).
   - $\pi_j = \frac{N_j}{n}$ (updated priors).
   - $\mu_j = \frac{1}{N_j} \sum_{i=1}^n \gamma_{ij} x_i$ (updated means).
   - $\Sigma_j = \frac{1}{N_j} \sum_{i=1}^n \gamma_{ij} (x_i - \mu_j)(x_i - \mu_j)^T$ (updated covariances).
4. **Convergence Check:**
   - Repeat E-step and M-step until $|\ln L^{(t)} - \ln L^{(t-1)}| \le \text{tol}$ or max iterations is reached.

---

### 10. What is cluster variance?
**Cluster variance** (also called *Within-Cluster Sum of Squares (WCSS)* or *inertia*) measures the total compactness or spread of data points around their assigned centroids:
$$\text{Var} = \sum_{j=1}^k \sum_{x_i \in C_j} \|x_i - \mu_j\|^2$$
- Lower cluster variance indicates tighter, denser clusters.
- Variance decreases monotonically as $k$ increases, reaching 0 when $k = n$.

---

### 11. What is the mountain/elbow method?
The **elbow method** is a visual heuristic used to choose the optimal number of clusters $k$ for partitioning methods like K-means:
1. Run K-means for a range of cluster counts (e.g. $k = 1, 2, \dots, 10$).
2. Compute total intra-cluster variance (inertia) for each $k$.
3. Plot variance vs. $k$.
4. Identify the **"elbow"** — the inflection point where the curve bends and the rate of variance reduction sharply diminishes. Beyond this point, adding clusters provides little benefit while increasing model complexity.

---

### 12. What is the Bayesian Information Criterion (BIC)?
The **Bayesian Information Criterion (BIC)** is a formal statistical criterion for model selection that balances model fit with model complexity:
$$\text{BIC} = p \ln(n) - 2 \ln(\hat{L})$$
- $p$: Number of independent free parameters in the model.
  - For a GMM with $k$ clusters in $d$ dimensions:
    $$p = k\left(1 + d + \frac{d(d+1)}{2}\right) - 1$$
- $n$: Number of data points.
- $\hat{L}$: Maximized likelihood of the model (with $\ln(\hat{L})$ as log-likelihood).
- **Rule:** A **lower BIC value indicates a better model** (high likelihood with minimal parameter overhead).

---

### 13. How to determine the correct number of clusters?
Several complementary methods exist:
1. **Elbow Method:** Locate the bend on a plot of intra-cluster variance vs. $k$.
2. **Bayesian Information Criterion (BIC) / AIC:** Find the minimum BIC score for probabilistic models like GMMs.
3. **Silhouette Analysis:** Measures how well-separated clusters are (scores from $-1$ to $+1$; higher is better).
4. **Gap Statistic:** Compares within-cluster dispersion against a null uniform reference distribution.
5. **Dendrogram Inspection:** In hierarchical clustering, find the largest vertical distance without cross-branches.
6. **Domain Knowledge:** Real-world requirements (e.g. business categories, physical constraints).

---

### 14. What is Hierarchical clustering?
**Hierarchical clustering** is a family of clustering algorithms that builds a nested hierarchy (tree) of clusters rather than a single flat partition.
- It **does not require specifying $k$ in advance**.
- Visualized as a **dendrogram** (tree diagram).
- Flat clusters can be obtained at any desired granularity by cutting the dendrogram horizontally at a chosen threshold.

---

### 15. What is Agglomerative clustering?
**Agglomerative clustering** is the standard "bottom-up" approach to hierarchical clustering:
1. **Start:** Each observation begins as its own individual cluster ($n$ singleton clusters).
2. **Merge:** At each step, the two closest clusters (according to a chosen linkage metric) are merged together into a single cluster.
3. **Finish:** Merging continues iteratively until all points are united into a single root cluster.

---

### 16. What is Ward's method?
**Ward's method** (minimum variance linkage) is an agglomerative linkage criterion that merges clusters based on variance minimization:
- At each step, it merges the two clusters that produce the **smallest increase in total within-cluster variance**:
  $$\Delta \text{ESS}_{AB} = \frac{n_A n_B}{n_A + n_B} \|\mu_A - \mu_B\|^2$$
- Strongly favors creating compact, spherical, balanced clusters and avoids chaining artifacts.

---

### 17. What is Cophenetic distance?
The **cophenetic distance** between two observations $x_i$ and $x_j$ is the inter-cluster distance at which they are first merged into the same cluster in a dendrogram.
- **Cophenetic Correlation Coefficient (CPCC):** The Pearson correlation between the original pairwise Euclidean distances and the cophenetic distances.
- A CPCC close to $1.0$ indicates that the dendrogram faithfully preserves the true pairwise geometry of the dataset.

---

### 18. What is scikit-learn?
**scikit-learn** is the standard, production-grade open-source machine learning library for Python, built on NumPy, SciPy, and Matplotlib.
- Provides unified APIs (`fit`, `predict`, `fit_predict`, `transform`) for:
  - Clustering (`KMeans`, `GaussianMixture`, `AgglomerativeClustering`).
  - Classification, regression, dimensionality reduction (PCA), model evaluation, and preprocessing.

---

### 19. What is scipy?
**SciPy** is the foundational scientific and numerical computing library for Python, built on NumPy.
- Provides specialized scientific routines for numerical optimization, linear algebra, integration, signal processing, and statistics.
- The `scipy.cluster.hierarchy` module provides essential tools for hierarchical clustering, including `linkage` (Ward, average, single, complete), `dendrogram` visualization, `fcluster` (cutting the tree), and `cophenet` (evaluating distance preservation).



In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

## 0. Initialize K-means centroids

**Signature:**
```python
def initialize(X, k):
```

### The Big Picture & Real-World Analogy
Imagine you are opening $k = 3$ new pizza delivery hubs across a sprawling city. Before you look at where all your customers live or start calculating delivery routes, you have to place your initial pins somewhere on the map. You wouldn't place a pin in the middle of the ocean or outside city borders—you look at the northernmost, southernmost, easternmost, and westernmost corners of the city, draw a **bounding box** around them, and drop $k$ pins randomly inside that box.

In K-means, this is exactly what `initialize(X, k)` does: it finds the minimum and maximum boundaries for each feature across the dataset and draws $k$ candidate starting centers uniformly at random inside that box.

---

### Step-by-Step Numerical Walkthrough
Suppose we have a tiny 2D dataset with $n = 3$ points:
- $P_1 = (2.0, 4.0)$
- $P_2 = (6.0, 2.0)$
- $P_3 = (4.0, 8.0)$

We want to initialize $k = 2$ centroids.

1. **Find Feature Boundaries (The Bounding Box):**
   - **Feature 1 ($x$-axis):** $\min = \min(2, 6, 4) = 2.0$, $\max = \max(2, 6, 4) = 6.0$. Range = $[2.0, 6.0]$.
   - **Feature 2 ($y$-axis):** $\min = \min(4, 2, 8) = 2.0$, $\max = \max(4, 2, 8) = 8.0$. Range = $[2.0, 8.0]$.
2. **Draw $k$ Random Points:**
   - Centroid 1 might be randomly sampled as: $C_1 = (3.5, 5.0)$
   - Centroid 2 might be randomly sampled as: $C_2 = (5.2, 3.1)$
   - Both points are guaranteed to fall within the bounding box $[2.0, 6.0] \times [2.0, 8.0]$!

---

### Key Lines Explained
```python
low, high = X.min(axis=0), X.max(axis=0)
return np.random.uniform(low, high, size=(k, d))
```
- `X.min(axis=0)` and `X.max(axis=0)`: The `axis=0` parameter tells NumPy to scan down the rows, computing the minimum and maximum for each feature column individually. For 2D data, `low` and `high` each have shape `(2,)`.
- `np.random.uniform(low, high, size=(k, d))`: Draws numbers from a continuous uniform distribution. Because `low` and `high` are arrays, NumPy automatically **broadcasts** them across the $k$ rows so each feature respects its own minimum and maximum boundaries.

---

### Where This Fits in the Pipeline
This is step 0: it provides the initial centroids $C$ needed by Task 1 ([1-kmeans.py](./1-kmeans.py)) to kick off the iterative clustering loop.

> **Student Tip:** While uniform sampling inside the bounding box is simple, it can occasionally place centroids far from any actual data points (or even leave some clusters empty). In production, libraries use smarter initialization strategies like **k-means++** (which we'll see in Task 10).



In [ ]:
initialize = __import__('0-initialize').initialize

np.random.seed(0)
a = np.random.multivariate_normal([30, 40], [[16, 0], [0, 16]], size=50)
b = np.random.multivariate_normal([10, 25], [[16, 0], [0, 16]], size=50)
c = np.random.multivariate_normal([40, 20], [[16, 0], [0, 16]], size=50)
d = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=50)
e = np.random.multivariate_normal([20, 70], [[16, 0], [0, 16]], size=50)
X = np.concatenate((a, b, c, d, e), axis=0)
np.random.shuffle(X)
plt.scatter(X[:, 0], X[:, 1], s=10)
plt.show()
print(initialize(X, 5))

## 1. K-means (from scratch)

**Signature:**
```python
def kmeans(X, k, iterations=1000):
```

### The Big Picture & Real-World Analogy
Think of K-means as a **dance between assignments and centers**:
1. **The Assignment Step:** Every dancer (data point) looks around the ballroom and joins the group of whichever dance instructor (centroid) is standing closest to them.
2. **The Update Step:** Each dance instructor moves to the exact middle (arithmetic mean) of the dancers who joined their group.
3. **Repeat:** Now that the instructors moved, some dancers might realize another instructor is now closer! They switch groups, instructors adjust their positions again, and the process repeats until nobody changes groups (convergence).

---

### Step-by-Step Numerical Walkthrough
Let's trace **one full iteration** by hand with $n = 4$ points in 2D:
- $A = (1, 1)$, $B = (2, 1)$, $C = (8, 8)$, $D = (9, 9)$
- Suppose our initial centroids are:
  - $C_1 = (1, 2)$
  - $C_2 = (7, 6)$

#### Step 1: Compute Distances & Assign Points
Calculate the Euclidean distance $d = \sqrt{(x - c_x)^2 + (y - c_y)^2}$:
- For point $A(1, 1)$:
  - $\text{dist}(A, C_1) = \sqrt{(1-1)^2 + (1-2)^2} = \sqrt{0 + 1} = 1.0$
  - $\text{dist}(A, C_2) = \sqrt{(1-7)^2 + (1-6)^2} = \sqrt{36 + 25} = \sqrt{61} \approx 7.81$
  - Winner: $A$ joins Cluster 1!
- For point $B(2, 1)$:
  - $\text{dist}(B, C_1) = \sqrt{(2-1)^2 + (1-2)^2} = \sqrt{1 + 1} = 1.41$
  - $\text{dist}(B, C_2) = \sqrt{(2-7)^2 + (1-6)^2} = \sqrt{25 + 25} \approx 7.07$
  - Winner: $B$ joins Cluster 1!
- For points $C(8, 8)$ and $D(9, 9)$:
  - Both are much closer to $C_2(7, 6)$ than to $C_1(1, 2)$.
  - Winner: $C$ and $D$ join Cluster 2!

#### Step 2: Recalculate Centroids (The Mean)
- New $C_1 = \text{mean}(A, B) = \left(\frac{1 + 2}{2}, \frac{1 + 1}{2}\right) = (1.5, 1.0)$
- New $C_2 = \text{mean}(C, D) = \left(\frac{8 + 9}{2}, \frac{8 + 9}{2}\right) = (8.5, 8.5)$

#### Step 3: Check Convergence
If in the next iteration points $A, B$ still belong to $C_1$ and $C, D$ still belong to $C_2$, the centroids won't move $\implies$ **Convergence reached!**

---

### Key Lines Explained
```python
distances = np.linalg.norm(X[:, np.newaxis] - C, axis=2)
clss = np.argmin(distances, axis=1)
for j in range(k):
    mask = clss == j
    C[j] = X[mask].mean(axis=0) if mask.any() else np.random.uniform(low, high)
if np.array_equal(C, C_prev):
    break
```
- `X[:, np.newaxis] - C`: Creates a 3D array of shape `(n, k, d)`. It broadcasts each of the $n$ points against all $k$ centroids.
- `np.linalg.norm(..., axis=2)`: Computes the Euclidean distance across the feature dimension $d$, yielding an `(n, k)` distance matrix.
- `np.argmin(distances, axis=1)`: For each point, finds the index $j \in \{0, \dots, k-1\}$ of the closest centroid.
- `X[mask].mean(axis=0)`: Computes the new center as the average coordinates of all points in cluster $j$. If a cluster happens to be empty (`not mask.any()`), it is reseeded randomly within data bounds.
- `np.array_equal(C, C_prev)`: Stops the loop early once centroids stabilize.

---

### Where This Fits in the Pipeline
This function returns the final centroid positions `C` and point labels `clss`. It is directly used to measure clustering quality in Task 2 ([2-variance.py](./2-variance.py)), to find the best $k$ in Task 3, and to seed the initial means for GMMs in Task 4.



In [ ]:
kmeans = __import__('1-kmeans').kmeans

np.random.seed(0)
a = np.random.multivariate_normal([30, 40], [[16, 0], [0, 16]], size=50)
b = np.random.multivariate_normal([10, 25], [[16, 0], [0, 16]], size=50)
c = np.random.multivariate_normal([40, 20], [[16, 0], [0, 16]], size=50)
d = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=50)
e = np.random.multivariate_normal([20, 70], [[16, 0], [0, 16]], size=50)
X = np.concatenate((a, b, c, d, e), axis=0)
np.random.shuffle(X)
C, clss = kmeans(X, 5)
print(C)
plt.scatter(X[:, 0], X[:, 1], s=10, c=clss)
plt.scatter(C[:, 0], C[:, 1], s=50, marker='*', c=list(range(5)))
plt.show()

## 2. Intra-cluster variance

**Signature:**
```python
def variance(X, C):
```

### The Big Picture & Real-World Analogy
Imagine inspecting a set of tidy closets. If all the shirts are hanging right next to each other on their rack, the closet has **low dispersion**. But if clothes are flung all across the floor far from the rack, the closet has **high dispersion**.

In clustering, **intra-cluster variance** (also known as **inertia** or **Within-Cluster Sum of Squares - WCSS**) measures how compact and tight your clusters are. For every single point, we measure how far it sits from its assigned centroid, square that distance (to penalize far outliers and eliminate negative signs), and add them all up. **Lower variance means tighter, better-grouped clusters!**

---

### Step-by-Step Numerical Walkthrough
Let's compute the intra-cluster variance by hand using our 4 points and the final centroids from Task 1:
- Points: $A(1, 1), B(2, 1), C(8, 8), D(9, 9)$
- Centroids: $C_1 = (1.5, 1.0)$ and $C_2 = (8.5, 8.5)$

#### 1. Cluster 1 Points ($A$ and $B$ assigned to $C_1$):
- Squared distance for $A(1, 1)$:
  $$\text{dist}^2(A, C_1) = (1 - 1.5)^2 + (1 - 1.0)^2 = (-0.5)^2 + 0^2 = 0.25$$
- Squared distance for $B(2, 1)$:
  $$\text{dist}^2(B, C_1) = (2 - 1.5)^2 + (1 - 1.0)^2 = (0.5)^2 + 0^2 = 0.25$$
- Cluster 1 Sum of Squared Errors: $0.25 + 0.25 = 0.50$

#### 2. Cluster 2 Points ($C$ and $D$ assigned to $C_2$):
- Squared distance for $C(8, 8)$:
  $$\text{dist}^2(C, C_2) = (8 - 8.5)^2 + (8 - 8.5)^2 = (-0.5)^2 + (-0.5)^2 = 0.25 + 0.25 = 0.50$$
- Squared distance for $D(9, 9)$:
  $$\text{dist}^2(D, C_2) = (9 - 8.5)^2 + (9 - 8.5)^2 = (0.5)^2 + (0.5)^2 = 0.25 + 0.25 = 0.50$$
- Cluster 2 Sum of Squared Errors: $0.50 + 0.50 = 1.00$

#### 3. Total Intra-Cluster Variance:
$$\text{Total Variance} = 0.50 + 1.00 = 1.50$$

---

### Key Lines Explained
```python
distances_sq = np.sum((X[:, np.newaxis] - C) ** 2, axis=2)
return np.sum(np.min(distances_sq, axis=1))
```
- `(X[:, np.newaxis] - C) ** 2`: Calculates the squared difference along each coordinate axis between all points and all centroids.
- `np.sum(..., axis=2)`: Sums across coordinate axes $x, y, \dots$ to get the squared Euclidean distance matrix of shape `(n, k)`.
- `np.min(..., axis=1)`: For each data point, picks the smallest squared distance (i.e. distance to its *nearest* centroid).
- `np.sum(...)`: Adds up these minimum squared distances across all $n$ points to produce a single variance scalar.

---

### Where This Fits in the Pipeline
Variance is the universal score used by Task 3 ([3-optimum.py](./3-optimum.py)) to test different values of $k$ and plot the **elbow curve**.



In [ ]:
variance = __import__('2-variance').variance

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
C = np.random.randint(0, 4, X.shape[0])
print(variance(X, C))

## 3. Optimum number of clusters (elbow method)

**Signature:**
```python
def optimum_k(X, kmin=1, kmax=None, iterations=1000):
```

### The Big Picture & Real-World Analogy
Suppose you are assigning delivery vans. If you have only 1 van ($k=1$), it travels huge distances and delivery variance is massive. If you buy a 2nd van ($k=2$), travel time drops dramatically. A 3rd van helps a little bit. But if you buy 100 vans for 100 customers ($k=n$), each customer has their own van and variance is literally 0—yet it costs a fortune!

In machine learning, **adding more clusters always decreases variance**. We don't want to just pick the lowest variance (which would trivially be $k = n$); we want the **sweet spot of diminishing returns** where adding one more cluster stops giving substantial improvement. On a graph of variance versus $k$, this point looks like a sharp bend or an **"elbow"**.

---

### Step-by-Step Numerical Walkthrough
Suppose we run K-means on a dataset for $k \in [1, 5]$ and record the variance:

| $k$ | Total Variance | Drop from previous $k$ ($\Delta \text{Var}$) | Interpretation |
| :---: | :---: | :---: | :--- |
| **1** | $1000$ | — | One massive cluster: very poor fit |
| **2** | $300$ | **$700$** | **Huge drop!** Data naturally splits into 2 groups |
| **3** | $250$ | $50$ | Minor improvement |
| **4** | $220$ | $30$ | Diminishing returns |
| **5** | $200$ | $20$ | Diminishing returns |

- Between $k=1$ and $k=2$, variance plummets by **$700$**.
- Between $k=2$ and $k=3$, variance only drops by **$50$**.
- The "elbow" is at **$k = 2$**. Adding more clusters past 2 yields diminishing returns.

---

### Key Lines Explained
```python
for k in range(kmin, kmax + 1):
    C, clss = kmeans(X, k, iterations)
    results.append((C, clss))
    variances.append(variance(X, C))
d_vars = [variances[0] - v for v in variances]
```
- The function iterates through every candidate cluster count from `kmin` to `kmax`.
- For each $k$, it runs `kmeans`, stores the cluster centers and assignments, and records the resulting `variance`.
- `d_vars = [variances[0] - v for v in variances]`: Computes the variance reduction relative to the baseline ($k = k_{min}$). Plotting this curve reveals where the rate of gain levels off.

---

### Where This Fits in the Pipeline
This concludes the K-means section. Next, we transition from hard partitioning to **soft probabilistic clustering** with Gaussian Mixture Models (Tasks 4–9).



In [ ]:
optimum_k = __import__('3-optimum').optimum_k

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
results, d_var = optimum_k(X, kmax=10)
print(d_var)
for i, (C, c) in enumerate(results):
    print("{}: {}".format(i + 1, c))
plt.plot(range(1, 11), d_var, 'bo-')
plt.xlabel('Clusters')
plt.ylabel('Delta variance')
plt.title('Optimizing number of clusters')
plt.show()

## 4. Initialize GMM parameters

**Signature:**
```python
def initialize(X, k):
```

### The Big Picture & Real-World Analogy
K-means draws hard boundary lines between clusters: point $A$ belongs 100% to Cluster 1 and 0% to Cluster 2. But the real world is rarely that black-and-white. 

Think of a **Gaussian Mixture Model (GMM)** like placing **weather clouds** over the data instead of rigid boxes. Each cloud represents a cluster:
1. **$\pi$ (Priors):** "How big/heavy is each cloud?" (e.g. 50% of the weather is Cloud 1, 50% is Cloud 2).
2. **$\mu$ (Means):** "Where is the center of each cloud located?"
3. **$\Sigma$ (Covariances):** "What is the shape, orientation, and spread of each cloud?" (Is it a circle? A tilted ellipse?).

Before the Expectation-Maximization algorithm (Tasks 6–8) can fine-tune these clouds, it needs an intelligent **starting guess**.

---

### Step-by-Step Numerical Walkthrough
Suppose we have a 2D dataset ($d = 2$) and want to fit $k = 2$ Gaussian components:

1. **Priors ($\pi$):** Assume each cluster is equally likely at the start:
   $$\pi = \left[\frac{1}{2}, \frac{1}{2}\right] = [0.5, 0.5]$$
2. **Means ($\mu$):** Rather than guessing blind, run K-means (from Task 1) to find initial centers near dense data regions:
   $$\mu_1 = (1.5, 1.0), \quad \mu_2 = (8.5, 8.5)$$
3. **Covariances ($S$ or $\Sigma$):** Start by assuming every cloud is a standard circle/sphere with unit spread in each direction (identity matrix $I_2$):
   $$S_1 = \begin{pmatrix} 1.0 & 0.0 \\ 0.0 & 1.0 \end{pmatrix}, \quad S_2 = \begin{pmatrix} 1.0 & 0.0 \\ 0.0 & 1.0 \end{pmatrix}$$

As EM iterates in later tasks, it will stretch, squash, and rotate these circular covariance matrices into whatever ellipses best fit the actual data!

---

### Key Lines Explained
```python
m, _ = kmeans(X, k)
pi = np.ones(k) / k
S = np.tile(np.eye(d), (k, 1, 1))
```
- `kmeans(X, k)`: Reuses our Task 1 implementation to obtain centroid means `m` of shape `(k, d)`.
- `np.ones(k) / k`: Initializes equal prior probabilities $\pi_j = \frac{1}{k}$ of shape `(k,)`.
- `np.eye(d)`: Creates a $d \times d$ identity matrix ($1$s on the diagonal, $0$s elsewhere).
- `np.tile(..., (k, 1, 1))`: Replicates the identity matrix $k$ times along a new leading axis, giving initial covariances `S` of shape `(k, d, d)`.

---

### Where This Fits in the Pipeline
The outputs `(pi, m, S)` are the initial parameters passed to Task 6 (the E-step) and Task 8 (the full EM training loop).



In [ ]:
initialize_gmm = __import__('4-initialize').initialize

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
pi, m, S = initialize_gmm(X, 4)
print(pi)
print(m)
print(S)

## 5. Gaussian probability density function (PDF)

**Signature:**
```python
def pdf(X, m, S):
```

### The Big Picture & Real-World Analogy
Imagine standing on the summit of a smooth, bell-shaped mountain. If you stand right at the peak ($x = \mu$), the altitude (density) is at its maximum. As you walk away down the slopes in any direction, the elevation drops off rapidly following an exponential decay curve.

The **multivariate Gaussian Probability Density Function (PDF)** answers the question: *"If a cluster is centered at mean $m$ with shape/spread $S$, what is the likelihood density of observing data point $x$?"*

---

### Understanding the Math Without Panic
The multivariate normal formula is:
$$p(x) = \frac{1}{(2\pi)^{d/2} \sqrt{|\Sigma|}} \exp\left(-\frac{1}{2} (x - \mu)^T \Sigma^{-1} (x - \mu)\right)$$

Let's break down its two components:
1. **The Mahalanobis Distance $(x - \mu)^T \Sigma^{-1} (x - \mu)$:**
   - In 1D, distance from the mean in standard deviations is $\left(\frac{x - \mu}{\sigma}\right)^2$.
   - In multidimensional space with covariance $\Sigma$, the Mahalanobis distance measures how many standard deviations away $x$ is, taking into account correlations and differences in scale across features.
2. **The Normalizing Factor $\frac{1}{(2\pi)^{d/2} \sqrt{|\Sigma|}}$:**
   - Ensures the total volume under the entire bell curve integrates to exactly $1.0$.

---

### Step-by-Step Numerical Walkthrough (1D Example)
To see the numbers clearly, let's look at 1D with mean $\mu = 0$ and variance $\sigma^2 = 1$ (so $\sigma = 1$):
$$\text{Normalizing constant} = \frac{1}{\sqrt{2\pi}} \approx \frac{1}{2.5066} \approx 0.3989$$

- **At the center ($x = 0$):**
  $$\text{dist}^2 = 0 \implies p(0) = 0.3989 \times e^0 = 0.3989$$
- **At 1 standard deviation ($x = 1$):**
  $$\text{dist}^2 = 1^2 = 1 \implies p(1) = 0.3989 \times e^{-0.5} \approx 0.3989 \times 0.6065 \approx 0.2420$$
- **At 3 standard deviations ($x = 3$):**
  $$\text{dist}^2 = 3^2 = 9 \implies p(3) = 0.3989 \times e^{-4.5} \approx 0.3989 \times 0.0111 \approx 0.0044$$

Notice how sharply the probability collapses as points move further from the mean!

---

### Key Lines Explained
```python
det = np.linalg.det(S)
S_inv = np.linalg.inv(S)
diff = X - m
coeff = 1 / (((2 * np.pi) ** (d / 2)) * np.sqrt(det))
mahal = np.sum((diff @ S_inv) * diff, axis=1)
return np.maximum(coeff * np.exp(-0.5 * mahal), 1e-300)
```
- `np.linalg.det(S)` and `np.linalg.inv(S)`: Computes the determinant $|\Sigma|$ and the inverse matrix $\Sigma^{-1}$.
- `(diff @ S_inv) * diff`: Vectorized Mahalanobis distance. For each point $x_i$, it computes $(x_i - \mu)^T \Sigma^{-1} (x_i - \mu)$ without a single Python `for` loop!
- `np.maximum(..., 1e-300)`: A numerical safety floor. In high dimensions or far out in the tails, $\exp(-\dots)$ can underflow to float $0.0$. Taking $\ln(0)$ would cause a catastrophic `NaN` in later steps, so we clamp the floor to $10^{-300}$.

---

### Where This Fits in the Pipeline
This PDF evaluation is the engine of Task 6: it allows the Expectation step to calculate how likely every data point is under each Gaussian cluster.



In [ ]:
pdf = __import__('5-pdf').pdf

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
P = pdf(X, [30, 40], [[75, 5], [5, 75]])
print(P)
print("shape:", P.shape)
print("min value:", P.min())

## 6. Expectation step (E-step)

**Signature:**
```python
def expectation(X, pi, m, S):
```

### The Big Picture & Real-World Analogy
Think of the E-step as **applying Bayes' Rule** to answer: *"Given our current Gaussian clouds, what is the probability that point $x_i$ was generated by Cluster $j$?"*

Imagine a clinic testing for two illnesses: Disease A (rare, prior 10%) and Disease B (common, prior 90%). When a patient shows specific symptoms $x$, the clinic combines the base rate (prior $\pi$) with the likelihood of those symptoms under each disease (the Gaussian PDF) to compute the **posterior probability** that the patient has Disease A versus B.

In GMMs, this posterior probability $\gamma_{ji}$ is called the **responsibility** of cluster $j$ for point $i$.

---

### Step-by-Step Numerical Walkthrough
Suppose we have a point $x_1$ and two clusters ($k = 2$):
- Cluster 1: prior $\pi_1 = 0.50$, Gaussian PDF likelihood $p(x_1 \mid \mu_1, \Sigma_1) = 0.08$
- Cluster 2: prior $\pi_2 = 0.50$, Gaussian PDF likelihood $p(x_1 \mid \mu_2, \Sigma_2) = 0.02$

#### 1. Calculate Weighted Likelihoods (Unnormalized Beliefs):
- For Cluster 1: $\pi_1 \times p_1(x_1) = 0.50 \times 0.08 = 0.04$
- For Cluster 2: $\pi_2 \times p_2(x_1) = 0.50 \times 0.02 = 0.01$

#### 2. Compute Total Evidence:
$$\text{Total} = 0.04 + 0.01 = 0.05$$

#### 3. Normalize to Get Posterior Probabilities ($\gamma$):
- Responsibility of Cluster 1: $\gamma_{1,1} = \frac{0.04}{0.05} = 0.80$ (80% chance)
- Responsibility of Cluster 2: $\gamma_{2,1} = \frac{0.01}{0.05} = 0.20$ (20% chance)
- Check: $0.80 + 0.20 = 1.00$ (sums to 100%). Point $x_1$ is **softly assigned**!

#### 4. Contribution to Overall Log-Likelihood:
$$\ln(\text{Total}) = \ln(0.05) \approx -2.996$$
Summing this value across all points gives the total log-likelihood $\ln L$.

---

### Key Lines Explained
```python
g = np.zeros((k, n))
for j in range(k):
    likelihood = pdf(X, m[j], S[j])
    g[j] = pi[j] * likelihood
total = g.sum(axis=0)
log_l = np.sum(np.log(total))
g /= total
return g, log_l
```
- `g[j] = pi[j] * likelihood`: Multiplies the cluster's prior probability by the Gaussian PDF for every point, filling row $j$.
- `total = g.sum(axis=0)`: Sums across clusters for each point to get the total marginal probability.
- `np.sum(np.log(total))`: The total log-likelihood of the dataset under the current model. This single number tells us how well the GMM fits the data.
- `g /= total`: Normalizes each column so the responsibilities for every data point sum to $1.0$.

---

### Where This Fits in the Pipeline
The responsibilities matrix `g` of shape `(k, n)` is passed directly into Task 7 (the M-step) to update the model parameters.



In [ ]:
initialize_gmm = __import__('4-initialize').initialize
expectation = __import__('6-expectation').expectation

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
pi, m, S = initialize_gmm(X, 4)
g, l = expectation(X, pi, m, S)
print(g)
print(np.sum(g, axis=0))
print(l)

## 7. Maximization step (M-step)

**Signature:**
```python
def maximization(X, g):
```

### The Big Picture & Real-World Analogy
In standard K-means, when updating a centroid, each point has an all-or-nothing vote ($1$ if it's in the cluster, $0$ if not).

In GMM's **Maximization step**, every point has a **weighted vote** equal to its responsibility $\gamma_{ji}$ from the E-step!
- If Alice is 80% in Cluster 1 and 20% in Cluster 2:
  - Cluster 1 counts 0.8 of Alice's coordinates when computing its new mean and variance.
  - Cluster 2 counts 0.2 of Alice's coordinates.

Given these soft assignments, the M-step updates the three parameters of each Gaussian cloud ($\pi, \mu, \Sigma$) to maximize the likelihood of the data.

---

### Step-by-Step Numerical Walkthrough
Suppose we have $n = 2$ points in 1D: $x_1 = 2.0$ and $x_2 = 10.0$.
Suppose Cluster 1's responsibilities are:
- For $x_1$: $\gamma_{1,1} = 0.9$
- For $x_2$: $\gamma_{1,2} = 0.1$

#### 1. Effective Number of Points ($N_1$):
$$N_1 = 0.9 + 0.1 = 1.0$$
Cluster 1 effectively "owns" $1.0$ data point in total.

#### 2. Updated Prior ($\pi_1$):
$$\pi_1 = \frac{N_1}{n} = \frac{1.0}{2} = 0.5$$

#### 3. Updated Mean ($\mu_1$):
$$\mu_1 = \frac{\gamma_{1,1} x_1 + \gamma_{1,2} x_2}{N_1} = \frac{(0.9 \times 2.0) + (0.1 \times 10.0)}{1.0} = \frac{1.8 + 1.0}{1.0} = 2.8$$
*Notice:* The new mean $\mu_1 = 2.8$ is pulled strongly toward $x_1 = 2.0$ (because weight was 0.9), with just a gentle tug toward $x_2 = 10.0$ (weight 0.1).

#### 4. Updated Covariance ($\Sigma_1$):
$$\Sigma_1 = \frac{0.9 \times (2.0 - 2.8)^2 + 0.1 \times (10.0 - 2.8)^2}{1.0} = 0.9 \times (-0.8)^2 + 0.1 \times (7.2)^2 = 0.9(0.64) + 0.1(51.84) = 0.576 + 5.184 = 5.76$$

---

### Key Lines Explained
```python
N = g.sum(axis=1)
pi = N / n
m = (g @ X) / N[:, np.newaxis]
for j in range(k):
    diff = X - m[j]
    S[j] = (g[j, :, np.newaxis] * diff).T @ diff / N[j]
```
- `N = g.sum(axis=1)`: Sums responsibilities across all $n$ data points to get the effective size $N_j$ for each cluster $j$.
- `pi = N / n`: The new mixing weights (priors), guaranteed to sum to $1.0$.
- `(g @ X) / N[:, np.newaxis]`: Matrix multiplication computes the weighted sum of points for all clusters at once, dividing each by $N_j$ to get updated centroid means.
- `(g[j, :, np.newaxis] * diff).T @ diff / N[j]`: Vectorized weighted outer product computing the updated $d \times d$ covariance matrix $\Sigma_j$.

---

### Where This Fits in the Pipeline
Together with Task 6 (E-step), the M-step forms the complete engine of Task 8 ([8-EM.py](./8-EM.py)), alternating back and forth until the model converges.



In [ ]:
initialize_gmm = __import__('4-initialize').initialize
expectation = __import__('6-expectation').expectation
maximization = __import__('7-maximization').maximization

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
pi, m, S = initialize_gmm(X, 4)
g, _ = expectation(X, pi, m, S)
pi, m, S = maximization(X, g)
print(pi)
print(m)
print(S)

## 8. Full EM algorithm

**Signature:**
```python
def expectation_maximization(X, k, iterations=1000, tol=1e-5, verbose=False):
```

### The Big Picture & Real-World Analogy
The **Expectation-Maximization (EM) algorithm** puts the entire Gaussian Mixture Model pipeline into motion. 

It is like an ongoing, polite conversation between two specialists:
- **The Statistician (E-step):** *"Given where our Gaussian clouds are currently placed, here is how much of every single point belongs to each cloud."*
- **The Cartographer (M-step):** *"Thank you. Given those fractional memberships, I have adjusted the centers, sizes, and rotations of the clouds to fit those points even better."*
- **The Statistician (E-step):** *"Great! Let me recalculate the memberships with the new clouds..."*

They repeat this back-and-forth cycle until the clouds stop shifting and the overall **log-likelihood plateaus**.

---

### The Convergence Story in Numbers
When we run EM on our synthetic dataset with $k = 4$ clusters:

```text
Log Likelihood after 0 iterations:  -652797.78665   <-- Initial K-means guess
Log Likelihood after 10 iterations:  -94855.45662   <-- Huge leap as clouds reshape
Log Likelihood after 20 iterations:  -94714.52057
Log Likelihood after 30 iterations:  -94590.87362
Log Likelihood after 40 iterations:  -94440.40559
Log Likelihood after 50 iterations:  -94439.93891
Log Likelihood after 52 iterations:  -94439.93889   <-- Plateau reached! Difference <= 1e-5
```

- **Monotonic Improvement:** Notice how the log-likelihood starts at $-652,797$ and increases (becomes less negative) with every iteration. This is a mathematical guarantee of the EM algorithm: **log-likelihood never gets worse**.
- **Early Stopping:** Between iteration 51 and 52, the change in log-likelihood was smaller than the tolerance $\text{tol} = 10^{-5}$. The algorithm detected that further iterations would produce negligible change and stopped automatically.

---

### Key Lines Explained
```python
pi, m, S = initialize(X, k)
g, log_l = expectation(X, pi, m, S)
for i in range(1, iterations + 1):
    pi, m, S = maximization(X, g)
    l_prev = log_l
    g, log_l = expectation(X, pi, m, S)
    if abs(log_l - l_prev) <= tol:
        break
```
1. `initialize(X, k)`: Sets up starting priors, means (via K-means), and spherical covariances.
2. `expectation(X, pi, m, S)`: Evaluates initial log-likelihood before any optimization.
3. `for i in range(1, iterations + 1)`: Alternates `maximization` followed by `expectation`.
4. `if abs(log_l - l_prev) <= tol: break`: Stops the loop as soon as the improvement between consecutive iterations is within `tol`.

---

### Where This Fits in the Pipeline
`expectation_maximization` is the complete training routine for a GMM. Task 9 ([9-BIC.py](./9-BIC.py)) will call it across various values of $k$ to automatically find the optimal number of Gaussian clusters.



In [ ]:
expectation_maximization = __import__('8-EM').expectation_maximization

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
k = 4
pi, m, S, g, l = expectation_maximization(X, k, 150, verbose=True)
print(X.shape[0] * pi)
print(m)
print(S)
print(l)
clss = np.sum(g * np.arange(k).reshape(k, 1), axis=0)
plt.scatter(X[:, 0], X[:, 1], s=10, c=clss)
plt.scatter(m[:, 0], m[:, 1], s=50, marker='*', c=list(range(k)))
plt.show()

## 9. Bayesian Information Criterion (BIC)

**Signature:**
```python
def BIC(X, kmin=1, kmax=None, iterations=1000, tol=1e-5, verbose=False):
```

### The Big Picture & Real-World Analogy
Imagine hiring contractors to model your company's sales data. 
- Contractor 1 uses a simple model with 2 parameters: it fits reasonably well.
- Contractor 2 uses a massive model with 10,000 parameters: it passes through every single data point perfectly! But it is wildly **overfitting**—memorizing random noise rather than discovering true patterns.

How do we fairly decide which model is best? **Occam's Razor:** *The simplest explanation that fits the data well is usually the best.*

The **Bayesian Information Criterion (BIC)** formalizes Occam's razor:
$$\text{BIC} = \underbrace{p \ln(n)}_{\text{Complexity Penalty}} - \underbrace{2 \ln(\hat{L})}_{\text{Goodness-of-Fit Reward}}$$
- As we add more clusters $k$, the log-likelihood $\ln(\hat{L})$ gets better (higher), which drives the score **down**.
- But each new cluster introduces more free parameters $p$, increasing the penalty $p \ln(n)$, which pushes the score **up**.
- **The best model is the one with the lowest BIC score!**

---

### Counting Free Parameters ($p$)
For a GMM with $k$ clusters in $d$ dimensions:
1. **Priors ($\pi$):** $k - 1$ free parameters (since they must sum to $1.0$).
2. **Means ($\mu$):** $k \times d$ parameters ($d$ coordinates per cluster center).
3. **Covariances ($\Sigma$):** $k \times \frac{d(d+1)}{2}$ parameters (covariance matrices are symmetric, so off-diagonals match).

$$\text{Total parameters: } p = k\left(1 + d + \frac{d(d+1)}{2}\right) - 1$$
For 2D data ($d = 2$), each cluster requires $1 + 2 + 3 = 6$ parameters:
$$p = 6k - 1$$

---

### Step-by-Step Numerical Walkthrough
For our dataset with $n = 12,500$ points in $d = 2$ dimensions, let's compare $k = 4$ versus $k = 5$:

#### At $k = 4$ clusters:
- Parameters: $p = 6(4) - 1 = 23$.
- Penalty term: $p \ln(n) = 23 \times \ln(12500) = 23 \times 9.4335 \approx 216.97$.
- Log-likelihood: $\ln(\hat{L}) = -94439.94 \implies -2 \ln(\hat{L}) \approx 188879.88$.
- $\text{BIC} = 216.97 + 188879.88 = \mathbf{189096.85}$.

#### At $k = 5$ clusters:
- Parameters: $p = 6(5) - 1 = 29$ ($+6$ extra parameters!).
- Penalty term: $29 \times 9.4335 \approx 273.57$ ($+56.6$ higher penalty!).
- Log-likelihood: $\ln(\hat{L}) = -94435.88 \implies -2 \ln(\hat{L}) \approx 188871.76$ (slight gain of $8.1$).
- $\text{BIC} = 273.57 + 188871.76 = \mathbf{189145.33}$.

**Conclusion:** Even though $k=5$ had a slightly higher log-likelihood, its BIC score is **higher** ($189,145 > 189,096$) because the tiny likelihood gain did not justify the extra 6 parameters. BIC correctly picks **$k = 4$** as the winner!

---

### Key Lines Explained
```python
p = k * (1 + d + d * (d + 1) // 2) - 1
bic[i] = p * np.log(n) - 2 * ll
best_idx = np.argmin(bic)
best_k = kmin + best_idx
```
- `p = k * (1 + d + d * (d + 1) // 2) - 1`: Computes the exact number of free parameters.
- `bic[i] = p * np.log(n) - 2 * ll`: Evaluates the BIC formula.
- `np.argmin(bic)`: Identifies the cluster index with the global minimum BIC score.

---

### Where This Fits in the Pipeline
This completes our from-scratch GMM implementation. Next, in Tasks 10 and 11, we will see how `scikit-learn` performs K-means and GMM in a single line of code!



In [ ]:
BIC = __import__('9-BIC').BIC

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
best_k, best_result, l, b_ic = BIC(X, kmin=1, kmax=10)
print(best_k)
print(best_result)
print(l)
print(b_ic)
plt.plot(range(1, 11), l, 'r-', label='Log Likelihood')
plt.xlabel('Clusters')
plt.legend()
plt.show()
plt.plot(range(1, 11), b_ic, 'b-', label='BIC')
plt.xlabel('Clusters')
plt.legend()
plt.show()

## 10. K-means (scikit-learn)

**Signature:**
```python
def kmeans(X, k):
```

### The Big Picture & Real-World Analogy
In Tasks 0 and 1, we built K-means with our own hands using basic NumPy operations. Building an algorithm from scratch is the best way to understand its inner workings. 

However, in professional production environments, we rely on **`scikit-learn`** (`sklearn`), the premier machine learning library in Python. Think of it like comparing a handcrafted prototype car to a commercial road vehicle:
- Our scratch version used uniform random initialization, which can occasionally pick poor starting points.
- `scikit-learn` uses **`k-means++`** initialization by default: an intelligent algorithm that spreads candidate centroids far apart from each other based on probability, leading to faster convergence and avoiding local minima.

---

### Step-by-Step Numerical Walkthrough
Suppose we have a dataset $X$ and want $k = 3$ clusters:

```python
# 1. Instantiate the model
model = sklearn.cluster.KMeans(n_clusters=3, random_state=0)

# 2. Fit the model to our data
model.fit(X)

# 3. Inspect the learned attributes
print("Centroids (C):\n", model.cluster_centers_)   # shape: (3, d)
print("Labels (clss):\n", model.labels_)            # shape: (n,)
```

- `model.cluster_centers_`: The coordinates of the converged centroid means (corresponds to `C` from Task 1).
- `model.labels_`: An array of integers $0, 1, 2$ indicating the cluster assignment for every point (corresponds to `clss` from Task 1).

---

### Key Lines Explained
```python
model = sklearn.cluster.KMeans(n_clusters=k)
model.fit(X)
return model.cluster_centers_, model.labels_
```
- `sklearn.cluster.KMeans(n_clusters=k)`: Configures the estimator with $k$ clusters.
- `model.fit(X)`: Runs the optimized Lloyd's algorithm in compiled Cython/C under the hood.
- `model.cluster_centers_` and `model.labels_`: Extracts the learned cluster centers and point memberships.

---

### Where This Fits in the Pipeline
This task bridges our custom from-scratch implementation with industry-standard machine learning workflows.



In [ ]:
kmeans_sklearn = __import__('10-kmeans').kmeans

np.random.seed(0)
a = np.random.multivariate_normal([30, 40], [[16, 0], [0, 16]], size=50)
b = np.random.multivariate_normal([10, 25], [[16, 0], [0, 16]], size=50)
c = np.random.multivariate_normal([40, 20], [[16, 0], [0, 16]], size=50)
d = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=50)
e = np.random.multivariate_normal([20, 70], [[16, 0], [0, 16]], size=50)
X = np.concatenate((a, b, c, d, e), axis=0)
np.random.shuffle(X)
C, clss = kmeans_sklearn(X, 5)
print(C)
plt.scatter(X[:, 0], X[:, 1], s=10, c=clss)
plt.scatter(C[:, 0], C[:, 1], s=50, marker='*', c=list(range(5)))
plt.show()

## 11. Gaussian Mixture Model (scikit-learn)

**Signature:**
```python
def gmm(X, k):
```

### The Big Picture & Real-World Analogy
Just as Task 10 streamlined K-means, Task 11 demonstrates how `scikit-learn`'s `GaussianMixture` implements the entire GMM pipeline (Tasks 4–9) in a clean, unified interface.

Under the hood, `sklearn.mixture.GaussianMixture` runs:
1. `k-means++` initialization for means.
2. The full Expectation-Maximization loop.
3. Numerical stabilization checks (such as adding a tiny regularizer `reg_covar = 1e-6` to covariance diagonals to prevent singular matrices).
4. Direct calculation of the model's **BIC** and **AIC** scores via built-in methods.

---

### Step-by-Step Numerical Walkthrough
Given a dataset $X$ and $k = 4$ clusters:

```python
model = sklearn.mixture.GaussianMixture(n_components=4)
model.fit(X)

pi = model.weights_        # Priors: array of shape (4,), e.g. [0.799, 0.080, 0.061, 0.060]
m = model.means_           # Centroid means: array of shape (4, 2)
S = model.covariances_     # Covariance matrices: array of shape (4, 2, 2)
clss = model.predict(X)    # Hard cluster assignments (argmax of responsibilities)
bic = model.bic(X)         # BIC score: 189096.85
```

Notice how `model.weights_`, `model.means_`, and `model.covariances_` match the exact outputs our from-scratch [8-EM.py](./8-EM.py) calculated!

---

### Key Lines Explained
```python
model = sklearn.mixture.GaussianMixture(n_components=k)
model.fit(X)
clss = model.predict(X)
bic = model.bic(X)
return model.weights_, model.means_, model.covariances_, clss, bic
```
- `GaussianMixture(n_components=k)`: Defines the GMM estimator with $k$ components (defaulting to full covariance matrices).
- `model.fit(X)`: Trains the mixture model using EM.
- `model.predict(X)`: Returns the most likely cluster index for each point ($\arg\max_j \gamma_{ji}$).
- `model.bic(X)`: Computes the Bayesian Information Criterion score.

---

### Where This Fits in the Pipeline
Now that we have explored both centroid-based clustering (K-means) and distribution-based clustering (GMM), our final task explores a fundamentally different paradigm: **Hierarchical Tree-based Clustering** (Task 12).



In [ ]:
gmm = __import__('11-gmm').gmm

np.random.seed(11)
a = np.random.multivariate_normal([30, 40], [[75, 5], [5, 75]], size=10000)
b = np.random.multivariate_normal([5, 25], [[16, 10], [10, 16]], size=750)
c = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=750)
d = np.random.multivariate_normal([20, 70], [[35, 10], [10, 35]], size=1000)
X = np.concatenate((a, b, c, d), axis=0)
np.random.shuffle(X)
pi, m, S, clss, bic = gmm(X, 4)
print(pi)
print(m)
print(S)
print(bic)
plt.scatter(X[:, 0], X[:, 1], s=10, c=clss)
plt.scatter(m[:, 0], m[:, 1], s=50, marker='*', c=list(range(4)))
plt.show()

## 12. Agglomerative (hierarchical) clustering

**Signature:**
```python
def agglomerative(X, dist):
```

### The Big Picture & Real-World Analogy
Imagine building a **family tree** (genealogy) backwards:
1. You start with $n$ distinct individuals living in separate houses.
2. First, siblings move into common family households (the closest pairs merge).
3. Next, households merge into neighborhood clans.
4. Eventually, all branches connect back to one single common ancestral root.

This is **Agglomerative Hierarchical Clustering**. It is a **"bottom-up"** approach: every data point begins as its own tiny cluster. At each step, the two closest clusters are merged together until everything forms one large tree, called a **dendrogram**.

Unlike K-means or GMM, you do **not** need to pick $k$ up front! You build the entire tree first, and then decide where to "slice" the branches horizontally based on a maximum distance threshold `dist`.

---

### Step-by-Step Numerical Walkthrough
Let's cluster 3 points on a 1D line by hand:
- $A = 1.0$
- $B = 3.0$
- $C = 10.0$

#### Step 1: Initial Distance Matrix
- $\text{dist}(A, B) = |1 - 3| = 2.0$
- $\text{dist}(B, C) = |3 - 10| = 7.0$
- $\text{dist}(A, C) = |1 - 10| = 9.0$

#### Step 2: First Merge
The smallest distance is between $A$ and $B$ ($\text{dist} = 2.0$).
- Merge $A$ and $B$ into a new cluster $(AB)$.
- In the dendrogram, draw a horizontal bar connecting $A$ and $B$ at height **$2.0$**.

#### Step 3: Second Merge
Now we have 2 clusters: $(AB)$ and $(C)$.
- The distance between $(AB)$ and $(C)$ is approximately $8.0$.
- In the dendrogram, draw a bar connecting $(AB)$ and $(C)$ at height **$8.0$**.

#### Step 4: Slicing the Tree with `dist`
- If we choose a distance threshold **$\text{dist} = 5.0$**:
  - The merge at height $2.0$ happened (so $A$ and $B$ are together in Cluster 0).
  - The merge at height $8.0$ did NOT happen (exceeds $5.0$, so $C$ remains in Cluster 1).
  - Result: **2 clusters** formed!

---

### What is Ward's Method?
When deciding which two clusters to merge, how do we measure distance between groups?
- **Single Linkage:** Distance between the closest pair of points (prone to long, stringy chains).
- **Complete Linkage:** Distance between the furthest pair of points.
- **Ward's Method (Used Here):** Merges the two clusters that cause the **smallest increase in total within-cluster variance**. Ward's method favors creating compact, round, evenly-sized clusters and is the most widely used linkage method in data science.

---

### Key Lines Explained
```python
Z = scipy.cluster.hierarchy.linkage(X, method='ward')
scipy.cluster.hierarchy.dendrogram(Z, color_threshold=dist)
clss = scipy.cluster.hierarchy.fcluster(Z, t=dist, criterion='distance')
```
- `linkage(X, method='ward')`: Computes the hierarchical cluster tree $Z$ of shape `(n-1, 4)` using Ward's minimum variance criterion.
- `dendrogram(Z, color_threshold=dist)`: Renders the beautiful tree diagram. Any branch merging below `dist` is colored distinctly per cluster.
- `fcluster(Z, t=dist, criterion='distance')`: Cuts the dendrogram at the threshold `dist` and returns the flat cluster label for every data point.

---

### Summary of the 3 Clustering Paradigms
Congratulations on completing the entire clustering pipeline! Here is how the three approaches compare:

| Paradigm | How it works | When to use | Key Hyperparameter |
| :--- | :--- | :--- | :--- |
| **K-Means** (Tasks 0–3, 10) | Centroid-based hard partitioning | Large datasets, fast baseline, spherical clusters | Number of clusters $k$ (Elbow method) |
| **Gaussian Mixture Model** (Tasks 4–9, 11) | Distribution-based soft clustering | Overlapping groups, elliptical clusters, uncertainty estimates | Number of clusters $k$ (BIC score) |
| **Hierarchical / Agglomerative** (Task 12) | Tree-based bottom-up merging | Small-to-medium datasets, hierarchical structures, visual trees | Cut distance threshold `dist` |



In [ ]:
agglomerative = __import__('12-agglomerative').agglomerative

np.random.seed(0)
a = np.random.multivariate_normal([30, 40], [[16, 0], [0, 16]], size=50)
b = np.random.multivariate_normal([10, 25], [[16, 0], [0, 16]], size=50)
c = np.random.multivariate_normal([40, 20], [[16, 0], [0, 16]], size=50)
d = np.random.multivariate_normal([60, 30], [[16, 0], [0, 16]], size=100)
e = np.random.multivariate_normal([20, 70], [[16, 0], [0, 16]], size=100)
X = np.concatenate((a, b, c, d, e), axis=0)
np.random.shuffle(X)
clss = agglomerative(X, 100)
plt.scatter(X[:, 0], X[:, 1], s=10, c=clss)
plt.show()